<style>
.jp-RenderedHTMLCommon h1 {color:#ff9900;font-size:2.25em}.jp-RenderedHTMLCommon h2{color:#2563a8}
.jp-RenderedHTMLCommon blockquote{border-left:6px solid #ff9900;background:#fff7e8;padding:.6em 1em}
.jp-RenderedHTMLCommon table{font-size:.9em}.jp-RenderedHTMLCommon code{color:#9a3412}
</style>

# AWS Glue Classifiers & Crawlers
## Cataloging MovieLens data through the AWS Console

**Guided UI lesson · `gksdatalake` · theory plus console instructions**

> Goal: understand how classification becomes schema, then use a crawler to create and safely maintain Catalog metadata.


# Learning outcomes

You will be able to:

- explain classifiers before configuring a crawler;
- choose between built-in and custom classifiers;
- create a CSV classifier suitable for MovieLens;
- create a Glue Catalog database;
- configure and run an S3 crawler through the AWS Console;
- verify tables, columns, locations, formats, and classifications;
- predict how recrawls and schema-change policies affect existing tables;
- diagnose common grouping, permission, header, and type-inference problems.


# Scenario and assumed S3 layout

Treat the local MovieLens files as already uploaded here:

```text
s3://gksdatalake/bronze/movielens/
├── movies/
│   └── movies.csv
└── ratings/
    └── ratings.csv
```

We will create:

- database: `gks_movielens_bronze`
- classifier: `gks_movielens_csv_classifier`
- crawler: `gks_movielens_bronze_crawler`
- expected tables: one for `movies`, one for `ratings`

Names are suggested conventions; adapt them to your organization's standards.


# Before the UI: choose the Region

Glue resources and the Data Catalog are **regional**. Select the intended AWS Region before creating anything and keep it selected throughout the exercise.

Confirm:

- the S3 bucket and the selected Region satisfy your architecture;
- you are in the expected AWS account;
- your console identity can create Glue databases, classifiers, crawlers, and pass the crawler role;
- the crawler role can list the bucket and read the two prefixes;
- if objects use a customer-managed KMS key, the role can decrypt them.

Changing Region later may make resources appear to have “disappeared.”


# 1 — What is a classifier?

A classifier examines data and answers two linked questions:

1. **Do I recognize this format?**
2. **If so, what schema and classification should describe it?**

Its output includes a classification label and inferred schema. During a crawl, custom classifiers are evaluated first in the order attached to the crawler. A fully confident match can determine the schema; otherwise Glue may continue evaluating classifiers and use the strongest result.

The classifier does not move, clean, or transform data. It supplies interpretation to the crawler.


# Built-in classifier families

Glue includes recognition for many common formats. Important families include:

- delimited text such as **CSV**;
- **JSON**, XML, and Amazon Ion;
- columnar/self-describing formats such as **Parquet**, ORC, and Avro;
- binary JSON (BSON);
- common web/server log patterns;
- supported relational/JDBC data sources.

Self-describing formats carry schema metadata with the file. CSV requires inference from delimiter, header, quoting, and sampled values, so it deserves more care.

[AWS: built-in and custom classifiers](https://docs.aws.amazon.com/glue/latest/dg/add-classifier.html)


# Custom classifier types

| Type | Use it when | Key definition |
|---|---|---|
| **CSV** | Delimited text needs deterministic delimiter/header/quote handling | delimiter, quote symbol, header behavior, optional column names, SerDe |
| **JSON** | Records are nested or located under a known object/array | JSONPath locating the record structure |
| **XML** | Repeating records sit under a specific element | row tag |
| **Grok** | Line-oriented text/logs follow a parseable pattern | Grok pattern plus optional custom patterns |

Custom classifiers are account/Region resources. Create them **before** the crawler so they can be attached in the intended order.


# How classifier selection works

```text
File/sample
   ↓
Custom classifier 1 → recognized with full certainty? → use result
   ↓ no
Custom classifier 2 → recognized with full certainty? → use result
   ↓ no
Built-in classifiers → choose a recognized/highest-certainty result
   ↓ none
UNKNOWN
```

Ordering matters when multiple custom patterns could match. Make classifiers narrow enough to avoid accidental matches.

> Editing a classifier does not automatically reclassify data already remembered by an existing crawler. For a clean correction, AWS recommends using a new crawler with the updated classifier.


# MovieLens CSV: inspect before classifying

`movies.csv`

```csv
movieId,title,genres
1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
```

`ratings.csv`

```csv
userId,movieId,rating,timestamp
1,1,4.0,964982703
```

Observations: comma delimiter, first-row headers, no multiline records expected, pipe-delimited values inside `genres`, decimal ratings, and epoch seconds in `timestamp`. Some movie titles can contain commas, so CSV quote handling matters.


# 2 — Create the custom CSV classifier

In the AWS Glue Console, open **Data Catalog → Classifiers**, then add a classifier.

Recommended intent:

- **Name:** `gks_movielens_csv_classifier`
- **Type:** CSV
- **Delimiter:** comma
- **Quote symbol:** double quote
- **Contains header:** present / has headings
- **Header names:** allow the file header to supply names
- **Allow single column:** not required for these files
- **Classification:** a clear CSV-related label if the UI requests one
- **SerDe:** prefer the CSV-aware option when quoted fields must be respected; do not choose blindly

Console wording can change; configure the behavior, not a memorized screen position.


# SerDe choice: why it matters

A **SerDe** converts stored bytes into rows/columns for engines that use the Catalog table.

- `OpenCSVSerDe` is useful when CSV contains quoted fields.
- `LazySimpleSerDe` is commonly associated with basic delimited text and type inference.
- `None` lets crawler detection choose where supported.

AWS notes an important tradeoff: if you switch a Catalog table to `OpenCSVSerDe`, downstream behavior may require inferred column types to be stored as strings and cast during queries. Validate with the engine that will consume the table.

Movie titles can contain commas, so testing quoted-title rows is essential.


# What we expect the classifier to infer

| Table | Column | Likely meaning | Validate after crawl |
|---|---|---|---|
| movies | `movieId` | movie identifier | numeric vs string |
| movies | `title` | title and release year text | quoted commas preserved |
| movies | `genres` | pipe-separated multi-value text | remains one string column |
| ratings | `userId` | user identifier | numeric vs string |
| ratings | `movieId` | movie identifier | compatible with movies key |
| ratings | `rating` | 0.5-step rating | floating/decimal inference |
| ratings | `timestamp` | Unix epoch seconds | likely integer/big integer, not automatically a timestamp |

Inferred types are technical guesses from observed values—not a business contract.


# 3 — Create the Catalog database

Open **Data Catalog → Databases**, then create a database.

- **Name:** `gks_movielens_bronze`
- **Description:** raw MovieLens metadata for the bronze layer
- **Location:** optional; a database location can provide a default, but each crawler-created table should still be checked for its actual S3 location

The database is a namespace for metadata. It does not upload files, create an S3 prefix, or run a database engine.

Verify that the database appears in the current account and Region before continuing.


# 4 — Prepare the crawler IAM role

The crawler assumes an IAM role. At minimum, it needs Glue service permissions plus scoped access to inspect the source.

Conceptual S3 permissions:

- `s3:ListBucket` on `gksdatalake`, constrained to `bronze/movielens/*` where practical;
- `s3:GetObject` on `arn:aws:s3:::gksdatalake/bronze/movielens/*`;
- KMS decrypt permission when applicable;
- permissions needed to create/update Catalog metadata.

The console may create a role, or you may select a pre-created least-privilege role. The operator also needs permission to pass that role to Glue.


# 5 — Create the crawler: identity and source

Open **Data Catalog → Crawlers**, create a crawler, and configure:

- **Name:** `gks_movielens_bronze_crawler`
- **Data source:** S3
- **Source path:** `s3://gksdatalake/bronze/movielens/`
- **Subsequent crawls:** start with “crawl all sub-folders” for learning and reliable re-evaluation
- **Connection:** not required for ordinary S3 access
- **IAM role:** the prepared crawler role

The common parent path is convenient, but table grouping must be checked carefully after the first run.


# Source-path design determines table shape

The intended folders are sibling datasets:

```text
bronze/movielens/movies/   → movies table
bronze/movielens/ratings/  → ratings table
```

Crawler heuristics group compatible data into tables and partitions. If data at the same level has sufficiently similar schemas, results may be grouped unexpectedly.

Safer options when grouping is wrong:

- create separate S3 targets for the two prefixes;
- create separate crawlers per dataset;
- adjust the crawler's table grouping/table-level configuration;
- use include/exclude patterns to keep unrelated files out.

For teaching clarity, **two explicit S3 targets** are preferred.


# 6 — Attach classifiers in the right order

In crawler configuration, add the custom classifier:

1. `gks_movielens_csv_classifier`
2. built-in classifiers remain available as fallback

Use the custom CSV classifier because we want predictable delimiter, header, and quoting behavior—not because every CSV always needs a custom classifier.

If several custom classifiers are attached, place the most specific first. An overly broad Grok or CSV classifier can claim data intended for another classifier.


# 7 — Configure crawler output

- **Target database:** `gks_movielens_bronze`
- **Table prefix:** optional; use something like `ml_` only if naming collisions are possible
- **Schedule:** on demand for this lesson
- **Schema change policy:** choose intentionally; for the first discovery, normal update behavior is acceptable
- **Deleted objects:** decide whether missing source objects should delete/deprecate Catalog metadata
- **Partition behavior:** for partitioned datasets, decide whether partitions inherit the table schema

Review the summary before creating the crawler. Confirm Region, role, paths, classifier, database, and schedule.


# 8 — Run and observe

Start the crawler and monitor its state until it returns to **Ready**.

Inspect:

- last crawl status and duration;
- tables added/updated/deleted;
- crawler history;
- CloudWatch logs when the run fails or output is unexpected.

Expected first-run result: two Catalog tables associated with the movies and ratings prefixes. Names may reflect folder names plus any configured prefix.

> A successful crawler run means metadata processing completed. It does not prove that the inferred schema is semantically correct.


# 9 — Validate the `movies` table

Open the table and verify:

- database and table name;
- location equals `s3://gksdatalake/bronze/movielens/movies/`;
- classification is CSV-related;
- delimiter and SerDe settings are appropriate;
- columns are exactly `movieId`, `title`, `genres` (case may be normalized);
- the header is not appearing as a data record in downstream previews/queries;
- a quoted title containing a comma remains one `title` value;
- no accidental partition keys were created.

If the location is the common parent, revisit target granularity/grouping.


# 10 — Validate the `ratings` table

Verify:

- location equals `s3://gksdatalake/bronze/movielens/ratings/`;
- columns are `userId`, `movieId`, `rating`, `timestamp`;
- `rating` supports values such as `4.0` and `3.5`;
- `timestamp` is recognized as epoch numeric data unless you explicitly transform it;
- `movieId` is type-compatible with the movies table;
- record counts and sample interpretation are plausible in a downstream query engine.

Schema compatibility matters because the two tables will later be joined on `movieId`.


# How the crawler builds schema

```text
S3 targets
  ↓ enumerate permitted objects
Classifier chain
  ↓ recognize format + produce candidate schema
Crawler grouping heuristics
  ↓ decide table vs partition boundaries
Catalog write
  ↓ database → table → columns/properties/partitions
Consumer engines
  ↓ interpret files using Catalog metadata
```

Schema is influenced by samples, file consistency, classifier settings, folder layout, and crawler grouping—not just the first row of one file.


# Schema changes on later crawls

By default, a crawler can update Catalog schema to reflect its source. Common policies include:

- **Update in place:** allow detected changes to modify table metadata.
- **Add new columns only:** merge additions without broadly overwriting established schema.
- **Log / ignore changes:** record differences without changing the existing table.
- **Delete behavior:** delete, deprecate, or ignore table/partition metadata when source objects disappear, depending on configuration.

For curated/stable contracts, a conservative policy is safer. For rapidly evolving raw discovery, controlled updates may be appropriate.


# Guided schema-evolution demonstration

Do this only with a disposable copy or controlled lab prefix:

1. Record the current columns and types.
2. Add a new column such as `sourceSystem` to **every relevant CSV header and row** in the test dataset.
3. Run the crawler with normal update behavior; inspect the added column.
4. Set **add new columns only**; test a compatible addition and observe.
5. Set **log/ignore**; introduce another change and confirm the Catalog remains stable while differences are logged.
6. Restore the test data and decide which metadata is authoritative.

Never introduce inconsistent row widths into a shared production prefix merely to test inference.


# Changing a classifier is not a retroactive fix

AWS Glue remembers previously crawled data. Updating the classifier affects new classification work but does not guarantee that all previously processed objects will be reclassified.

When correcting a bad classifier/schema:

1. fix and validate the classifier;
2. create a **new crawler** using the corrected classifier;
3. target a test database or carefully named tables;
4. compare location, schema, SerDe, properties, and partitions;
5. promote the corrected metadata deliberately.

This avoids assuming that “edit classifier + rerun” fully resets history.


# Recrawl strategies

| Strategy | Behavior | Best fit |
|---|---|---|
| Crawl everything | Re-evaluate the full target | Small datasets, major schema changes, troubleshooting |
| New folders only | Discover newly added folder partitions | Stable append-only layouts |
| Event mode | Use S3 event signals to focus on changes | Large datasets needing faster incremental discovery |

Incremental modes improve efficiency but constrain what changes can be rediscovered. Choose based on the data-arrival contract, not merely speed.

[AWS: customizing crawler behavior](https://docs.aws.amazon.com/glue/latest/dg/crawler-configuration.html)


# Partitions: not used here, but essential

The current paths contain one dataset file per table and no partition keys. A partitioned layout might look like:

```text
ratings/year=2026/month=09/day=01/part-....csv
```

The crawler can register `year`, `month`, and `day` partitions so query engines can prune data. Ensure:

- all files beneath a table root share a compatible schema;
- partition folder naming is consistent;
- partition keys do not also conflict with columns inside the files;
- future partitions inherit stable table metadata where appropriate.


# Troubleshooting map

| Symptom | Likely cause | First check |
|---|---|---|
| Access denied | crawler role/KMS/bucket policy | role ARN and exact object prefix |
| Zero tables | wrong path, empty prefix, unsupported/encrypted data | S3 objects and crawl logs |
| One table instead of two | grouping heuristics/common parent target | explicit dataset targets |
| Header becomes data | classifier header configuration | first row + classifier settings |
| `col0`, `col1` names | header not recognized | custom CSV header behavior |
| All strings/wrong numeric types | limited or inconsistent samples | source consistency and explicit schema |
| Title splits at comma | quote/SerDe behavior | quoted sample record |
| Duplicate tables after changes | path/table-level or prefix changes | table locations and crawler history |


# Crawler boundaries and good practice

A crawler is excellent for **metadata discovery**, but it is not:

- a data-quality engine;
- an ETL transformation;
- a schema registry enforcing producer contracts;
- proof that every file conforms;
- a replacement for access governance;
- automatically the cheapest way to manage stable schemas.

For stable production data, consider explicit Catalog definitions through infrastructure as code. Use crawlers where discovery and partition maintenance provide real value.


# Operational checklist

- [ ] Correct AWS account and Region
- [ ] Expected S3 objects and folder boundaries
- [ ] Least-privilege crawler role, including KMS if needed
- [ ] Classifier created first and ordered correctly
- [ ] Two explicit S3 targets or verified grouping behavior
- [ ] Correct output database and naming prefix
- [ ] Intentional recrawl and schema-change policies
- [ ] Table location, SerDe, classification, columns, and partitions verified
- [ ] Logs/history reviewed
- [ ] Downstream sample query validates quoted titles and numeric types
- [ ] Schedule disabled or justified after the lesson


# What else matters?

Topics often missed in a first crawler exercise:

- **Lake Formation:** may add a separate governance/permission layer over Catalog data.
- **Cross-account access:** requires resource policies, trust, and data permissions—not only a crawler path.
- **Table versioning:** Catalog updates can create versions useful for auditing/recovery.
- **Exclusions:** keep temporary, checksum, malformed, or unrelated files out of a target.
- **Small-file/layout discipline:** crawlers catalog metadata; they do not compact files.
- **Cost:** crawlers are billed by usage; run on a justified event/schedule rather than continuously.
- **Consumer compatibility:** validate Athena/EMR/Redshift Spectrum behavior for the chosen SerDe and types.


# Cleanup after a temporary lab

If these are disposable resources, clean them up in dependency-aware order:

1. disable any crawler schedule;
2. delete the crawler;
3. delete the custom classifier if unused elsewhere;
4. delete the two Catalog tables;
5. delete the database if empty;
6. remove the lab IAM role/policies only if not shared;
7. retain or delete S3 data according to the course's data-retention plan.

Deleting Catalog tables does not normally delete the underlying S3 objects. Verify every target before destructive cleanup.


# Knowledge check

1. Why must the classifier be understood before the crawler?
2. Why are two explicit S3 targets safer for this folder layout?
3. What should `timestamp` mean after inference—and what does it not mean yet?
4. When should a crawler only add columns rather than overwrite schema?
5. Why might editing a classifier fail to repair previously crawled metadata?
6. What is the difference between classifier, crawler, Catalog table, and underlying CSV?
7. Which permissions belong to the console operator, and which belong to the crawler role?


# Summary

**Classifier → Crawler → Data Catalog**

- A classifier recognizes format and proposes schema.
- A crawler discovers objects, invokes classifiers, groups data, and writes metadata.
- A database namespaces the resulting tables.
- Table definitions describe S3 data; they do not contain it.
- Later crawls can update metadata, so schema-change and recrawl policies are architectural decisions.
- Always validate inferred types, quoting, table locations, grouping, and permissions.

## Official references

- [Using crawlers to populate the Data Catalog](https://docs.aws.amazon.com/glue/latest/dg/add-crawler.html)
- [Defining and managing classifiers](https://docs.aws.amazon.com/glue/latest/dg/add-classifier.html)
- [Preventing crawler schema changes](https://docs.aws.amazon.com/glue/latest/dg/crawler-schema-changes-prevent.html)
- [Supported crawler data sources](https://docs.aws.amazon.com/glue/latest/dg/crawler-data-stores.html)
